# Clase 20 · Clonación de voz con Qwen3-TTS

**Objetivo:** generar voz sintética en español a partir de una grabación de referencia propia y comparar dos modos de extracción de la voz.

**Estructura:** cargar una única instancia del modelo en CPU → recortar el silencio inicial de la referencia → generar una muestra → comparar `x_vector_only_mode=False` y `x_vector_only_mode=True`.

- **Modelo:** [Qwen/Qwen3-TTS-12Hz-0.6B-Base](https://huggingface.co/Qwen/Qwen3-TTS-12Hz-0.6B-Base)
- **Documentación:** [Qwen3-TTS — Voice Clone](https://github.com/QwenLM/Qwen3-TTS#voice-clone)

Utiliza una grabación propia o una voz cuyo uso tengas autorizado. La primera carga descarga el modelo; la inferencia se configura para CPU.


In [3]:
from pathlib import Path
import os

import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

torch.set_num_threads(min(8, os.cpu_count() or 1))
device = "cpu"
dtype = torch.float32
MODELO_ID = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"
Path("outputs").mkdir(parents=True, exist_ok=True)
RUTA_AUDIO = Path("data/A1_clara.wav")
RUTA_REFERENCIA_RECORTADA = Path("outputs/referencia_A1_clara_recortada.wav")
TEXTO_REFERENCIA = "Hola soy Santi y quiero jugar al gta seis."  # Debe coincidir exactamente con la frase grabada en data/A1_clara.wav.
if not RUTA_AUDIO.is_file():
    raise FileNotFoundError(f"No se encontró el audio de referencia: {RUTA_AUDIO}")

# Una sola instancia reutilizada en todas las generaciones.
modelo_tts = Qwen3TTSModel.from_pretrained(
    MODELO_ID,
    device_map=device,
    dtype=dtype,
)


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 32.79it/s]


In [4]:
if not RUTA_AUDIO.is_file():
    raise FileNotFoundError(f"No se encontró el audio de referencia: {RUTA_AUDIO}")

referencia, sr_referencia = sf.read(RUTA_AUDIO, dtype="float32", always_2d=True)
referencia = referencia.mean(axis=1)
if referencia.size == 0 or not np.isfinite(referencia).all():
    raise ValueError(f"La referencia está vacía o contiene valores no válidos: {RUTA_AUDIO}")

# Detecta actividad por RMS en tramas de 20 ms y conserva 50 ms de contexto.
longitud_trama = max(1, round(sr_referencia * 0.020))
pico = float(np.max(np.abs(referencia)))
umbral = max(pico * 0.02, 1e-6)
longitud_rellenada = int(np.ceil(len(referencia) / longitud_trama) * longitud_trama)
relleno = np.pad(referencia, (0, longitud_rellenada - len(referencia)))
tramas = relleno.reshape(-1, longitud_trama)
rms_tramas = np.sqrt(np.mean(np.square(tramas), axis=1))
activas = np.flatnonzero(rms_tramas >= umbral)
if activas.size == 0:
    raise ValueError("No se detectó voz por encima del umbral de energía; revisa la grabación.")

margen = round(sr_referencia * 0.050)
inicio = max(0, int(activas[0] * longitud_trama) - margen)
fin = min(len(referencia), int((activas[-1] + 1) * longitud_trama) + margen)
referencia_recortada = referencia[inicio:fin]
sf.write(RUTA_REFERENCIA_RECORTADA, referencia_recortada, sr_referencia)
print(f"Referencia original: {len(referencia) / sr_referencia:.2f} s")
print(f"Referencia recortada: {len(referencia_recortada) / sr_referencia:.2f} s")
print(f"Audio listo: {RUTA_REFERENCIA_RECORTADA}")
display(Audio(referencia_recortada, rate=sr_referencia, normalize=False))


Referencia original: 6.00 s
Referencia recortada: 5.10 s
Audio listo: outputs\referencia_A1_clara_recortada.wav


## 1. Generar una muestra de clonación

`x_vector_only_mode=False` utiliza el embedding del hablante y el texto de la
referencia. Ajusta `TEXTO_REFERENCIA` para que coincida exactamente con lo que se
oye en el recorte.


In [5]:
TEXTO_NUEVO = "Quiero que Feid y Alvarito saquen EP ya."
RUTA_SALIDA = Path("outputs/act5_clonado.wav")

prompt_completo = modelo_tts.create_voice_clone_prompt(
    ref_audio=str(RUTA_REFERENCIA_RECORTADA),
    ref_text=TEXTO_REFERENCIA,
    x_vector_only_mode=False,
)
wavs, sr_salida = modelo_tts.generate_voice_clone(
    text=TEXTO_NUEVO,
    language="Spanish",
    voice_clone_prompt=prompt_completo,
    max_new_tokens=256,
)
sf.write(RUTA_SALIDA, wavs[0], sr_salida)
print(f"Audio generado: {RUTA_SALIDA} | {sr_salida} Hz")
display(Audio(str(RUTA_SALIDA)))


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Audio generado: outputs\act5_clonado.wav | 24000 Hz


## 2. Comparar los modos de clonación

Se usa el mismo texto a generar, la misma referencia recortada y una semilla fija
para que la escucha se centre en las diferencias entre los modos. La generación
puede tener variación propia del modelo, así que la semilla no implica una
reproducción idéntica entre versiones o equipos.


In [9]:
TEXTO_COMPARATIVA = "Como vuelva a ganar Pedrito, me voy a Andorra."
RUTA_MODO_COMPLETO = Path("outputs/act6_modo_completo.wav")
RUTA_MODO_VECTOR = Path("outputs/act6_modo_solo_vector.wav")
MAX_TOKENS_COMPARATIVA = 80
DURACION_MAX_S = 4.0


def limitar_duracion(senal, sr, max_segundos=4.0):
    """Recorta la señal a un máximo de segundos para eliminar colas de ruido o silencio."""
    arr = np.asarray(senal, dtype=np.float32).reshape(-1)
    n_max = max(1, int(round(sr * max_segundos)))
    return arr[:n_max]


# Modo A: embedding del hablante más transcripción de la referencia.
torch.manual_seed(42)
prompt_completo = modelo_tts.create_voice_clone_prompt(
    ref_audio=str(RUTA_REFERENCIA_RECORTADA),
    ref_text=TEXTO_REFERENCIA,
    x_vector_only_mode=False,
)
wav_completo, sr_completo = modelo_tts.generate_voice_clone(
    text=TEXTO_COMPARATIVA,
    language="Spanish",
    voice_clone_prompt=prompt_completo,
    max_new_tokens=MAX_TOKENS_COMPARATIVA,
)
audio_completo = limitar_duracion(wav_completo[0], sr_completo, DURACION_MAX_S)
sf.write(RUTA_MODO_COMPLETO, audio_completo, sr_completo)

# Modo B: solo embedding del hablante, sin transcripción de referencia.
torch.manual_seed(42)
prompt_solo_vector = modelo_tts.create_voice_clone_prompt(
    ref_audio=str(RUTA_REFERENCIA_RECORTADA),
    x_vector_only_mode=True,
)
wav_solo_vector, sr_vector = modelo_tts.generate_voice_clone(
    text=TEXTO_COMPARATIVA,
    language="Spanish",
    voice_clone_prompt=prompt_solo_vector,
    max_new_tokens=MAX_TOKENS_COMPARATIVA,
)
audio_solo_vector = limitar_duracion(wav_solo_vector[0], sr_vector, DURACION_MAX_S)
sf.write(RUTA_MODO_VECTOR, audio_solo_vector, sr_vector)

print(f"Modo completo ({len(audio_completo) / sr_completo:.2f} s): {RUTA_MODO_COMPLETO}")
display(Audio(str(RUTA_MODO_COMPLETO)))
print(f"Solo embedding ({len(audio_solo_vector) / sr_vector:.2f} s): {RUTA_MODO_VECTOR}")
display(Audio(str(RUTA_MODO_VECTOR)))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Modo completo (4.00 s): outputs\act6_modo_completo.wav


Solo embedding (3.92 s): outputs\act6_modo_solo_vector.wav


## Análisis comparativo de clonación

El silencio inicial se recorta antes de extraer la referencia para que el modelo
reciba principalmente voz y no una región sin información del hablante. Esto
reduce el contexto inútil y evita que una pausa larga ocupe parte de la ventana
analizada; se conserva un margen alrededor de la actividad detectada para no
cortar ataques de voz. El umbral es relativo al pico de la propia grabación y la
salida recortada se guarda aparte, sin modificar `data/A1_clara.wav`.

En `x_vector_only_mode=False`, el prompt combina el embedding de la voz con la
transcripción de referencia: si esta es correcta, puede aportar alineación y
ayudar a reproducir rasgos de pronunciación y prosodia. En `True`, se usa solo el
embedding del hablante y no se proporciona la transcripción; el contenido hablado
no condiciona ese prompt, por lo que la prosodia puede ser distinta o menos
específica. La calidad depende también de la claridad, duración y correspondencia
entre el audio y su transcripción. Escucha ambas salidas y registra cuál conserva
mejor identidad, naturalidad y entonación; no hay un modo universalmente superior.


## Interpretación de resultados (`x_vector_only_mode=False` vs. `True`)
* **`x_vector_only_mode=False`**: utiliza información adicional de la referencia (más allá del vector de identidad global del locutor), lo que permite transferir con mayor fidelidad matices de entonación, ritmo o estilo asociados al audio de referencia.
* **`x_vector_only_mode=True`**: se apoya principalmente en el *x-vector* (identidad/timbre del locutor), priorizando mantener el timbre de la voz objetivo con menor arrastre prosódico literal del segmento de referencia.
* **Comparativa auditiva**: [indica aquí qué diferencia notaste al escuchar ambos `<audio>` generados en cuanto a naturalidad, entonación y parecido con la voz de referencia].